In [ ]:
# Environment and path setup
import os, sys, platform

def find_repo_root():
    """Attempt to locate the repository root that contains the 'uraster' package."""
    candidates = [
        os.getcwd(),
        os.path.dirname(os.getcwd()),
        os.path.dirname(os.path.dirname(os.getcwd())),
    ]
    for cand in candidates:
        if os.path.isdir(os.path.join(cand, 'uraster')):
            return os.path.realpath(cand)
    return os.getcwd()

def find_data_folder():
    """Find the data/example_1 folder regardless of current working directory."""
    candidates = [
        os.path.join(os.getcwd(), 'data', 'example_1'),
        os.path.join(os.path.dirname(os.getcwd()), 'data', 'example_1'),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), 'data', 'example_1'),
    ]
    for p in candidates:
        if os.path.isdir(p):
            return os.path.realpath(p)
    raise FileNotFoundError('Could not locate data/example_1 folder from current directory.')

sPath_library = find_repo_root()
if sPath_library not in sys.path:
    sys.path.append(sPath_library)

sFolder_data = find_data_folder()
print(f'Data folder path: {sFolder_data}')

In [ ]:
# File paths
sFilename_source_mesh = os.path.join(sFolder_data, 'input', 'rhealpix_global_res3.geojson')
sFilename_raster = os.path.join(sFolder_data, 'input', 'EDGAR_CH4_emission_global_2015.tiff')

sFilename_target_mesh = os.path.join(sFolder_data, 'output', 'uraster.geojson')
sFilename_mesh_png = os.path.join(sFolder_data, 'output', 'mesh.jpg')
sFilename_raster_png = os.path.join(sFolder_data, 'output', 'raster.png')
sFilename_variable_png = os.path.join(sFolder_data, 'output', 'uraster.png')
sFilename_variable_animation = os.path.join(sFolder_data, 'output', 'uraster.gif')

In [ ]:
# Import uraster class
from uraster.classes.uraster import uraster

## Run Remapping and Visualization
The following cell sets up the configuration, runs remapping, reports inputs/outputs, and creates visualizations. Animation can be toggled via `iFlag_create_animation`.

In [ ]:
# Configure and run
aConfig = dict()
aConfig['sFilename_source_mesh'] = sFilename_source_mesh
aFilename_source_raster = [sFilename_raster]
aConfig['aFilename_source_raster'] = aFilename_source_raster
aConfig['sFilename_target_mesh'] = sFilename_target_mesh

pRaster = uraster(aConfig)
pRaster.setup()

pRaster.report_inputs()
# Focus on center of first raster extent
dLongitude_focus_in = (pRaster.aExtent_rasters[0] + pRaster.aExtent_rasters[2]) / 2
dLatitude_focus_in = (pRaster.aExtent_rasters[1] + pRaster.aExtent_rasters[3]) / 2
pRaster.visualize_source_mesh(
    sFilename_out=sFilename_mesh_png,
    dLongitude_focus_in=dLongitude_focus_in,
    dLatitude_focus_in=dLatitude_focus_in,
)
# pRaster.visualize_raster(sFilename_out=sFilename_raster_png)

pRaster.run_remap()
pRaster.report_outputs()

sColormap = 'terrain'
pRaster.visualize_target_mesh(
    sFilename_out=sFilename_variable_png,
    sColormap=sColormap,
    dLongitude_focus_in=dLongitude_focus_in,
    dLatitude_focus_in=dLatitude_focus_in,
)

# Optional animation (can take time)
pRaster.visualize_target_mesh(
    sFilename_out=sFilename_variable_animation,
    sColormap=sColormap,
    dLongitude_focus_in=dLongitude_focus_in,
    dLatitude_focus_in=dLatitude_focus_in,
    iFlag_create_animation=True,
    iAnimation_frames=360,
    sAnimation_format='mp4',
)

pRaster.cleanup()
print('done')